# 06 · Table IV: other generators

**Configuration note.** The four published rows were produced with an earlier
verifier configuration: a single-layer read (last-layer mean, d = 1,536, about
0.4M parameters) trained on **800-question pools**, not the three-layer
1.18M-parameter verifier of Tables I-III. Part A reproduces those rows from
their caches. Part B recomputes the Math-Instruct row with the three-layer
recipe on a 3,000-question pool for comparison.

| row | cache | generator |
|---|---|---|
| Distilled r16, r32 | `verifier_final` (r32) | Qwen2.5-1.5B + LoRA (Part C trains it) |
| Few-shot CoT | `verifier_fewshot` | Qwen2.5-1.5B, four exemplars |
| Math-Instruct | `verifier_mathmax` | Qwen2.5-Math-1.5B-Instruct, chat prompt |

In [ ]:
#@title Setup: mount Drive, install lvpp, set cache paths
import os, sys, json, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                    "lvpp[gpu,gbt] @ git+https://github.com/KARTIKPANSURIYA/lvpp.git"], check=True)

# Root of the cached candidates/features. The default is the Drive layout the
# paper's runs wrote to; set LVPP_CACHE to use another location.
CACHE = os.environ.get("LVPP_CACHE", "/content/drive/MyDrive/lcldd_runs")
RESULTS = os.environ.get("LVPP_RESULTS", os.path.join(CACHE, "lvpp_results"))
os.makedirs(RESULTS, exist_ok=True)
PATHS = dict(
    main_pool   = f"{CACHE}/verifier_upgrade/pool.pt",       # 7,473 GSM8K train questions x 8
    main_test   = f"{CACHE}/verifier_upgrade/testfeat.pt",   # 1,319 GSM8K test questions x 8
    prefix_root = f"{CACHE}/abort_main_draw",                # prefix draw: pool/ and test/ shards
    span_root   = f"{CACHE}/span_fixed_vllm",                # Fig. 2 draw (1.5B-Instruct, corrected trigger)
    mi_pool     = f"{CACHE}/pending/mi_pool.pt",             # Math-Instruct on GSM8K
    mi_test     = f"{CACHE}/pending/mi_test.pt",
    math_pool   = f"{CACHE}/pending/math_pool.pt",           # Math-Instruct on MATH
    math_test   = f"{CACHE}/pending/math_test.pt",
    x15_pool    = f"{CACHE}/novelty/x15_pool.pt",            # 1.5B-Instruct, 1,200 questions
    x3_pool     = f"{CACHE}/novelty/x3_pool.pt",             # 3B-Instruct, 1,200 questions
    x3_test     = f"{CACHE}/novelty/x3_test.pt",             # 3B-Instruct, 400 test questions
    legacy_fewshot = f"{CACHE}/verifier_fewshot",            # Table IV, single-layer verifier
    legacy_mathmax = f"{CACHE}/verifier_mathmax",
    distilled_r32  = f"{CACHE}/verifier_final",
)
import numpy as np, pandas as pd, torch
from sklearn.metrics import roc_auc_score
print("device:", "cuda" if torch.cuda.is_available() else "cpu", "| cache:", CACHE)

In [ ]:
def report(rows, expected, keys, tol=0.6):
    """Print measured vs published values. Regenerated candidates are a new
    random draw, so agreement within about half a point is expected; from the
    released caches the numbers should match to the reported precision."""
    out = []
    for k in keys:
        m, e = rows.get(k), expected.get(k)
        if m is None or e is None:
            flag = ""
        elif e < 1e-3:                       # p-values: significant at the same level
            flag = "ok" if m < 1e-3 else "CHECK"
        elif e <= 1.0:                       # AUCs and rates
            flag = "ok" if abs(m - e) <= 0.01 else "CHECK"
        else:                                # accuracies in percentage points
            flag = "ok" if abs(m - e) <= tol else "CHECK"
        out.append(dict(quantity=k, measured=None if m is None else round(m, 4), paper=e, status=flag))
    df = pd.DataFrame(out)
    print(df.to_string(index=False))
    return df

In [ ]:
#@title Part A - published rows from the single-layer caches (3 seeds)
from lvpp.caches import load_pool_legacy, load_dict_B, dict_B_as_pool
from lvpp.selection import selection_hits
from lvpp.verifier import fit_verifier
rows = {}
for name, key in [("few-shot CoT", "legacy_fewshot"), ("Math-Instruct", "legacy_mathmax"),
                  ("distilled r32", "distilled_r32")]:
    d = PATHS[key]
    if not os.path.exists(f"{d}/pool.pt"):
        print("missing:", d); continue
    P, Z, y = load_pool_legacy(f"{d}/pool.pt")
    te = load_dict_B(f"{d}/testfeat.pt", variants=("pooled",))
    accs = [100 * selection_hits(te, "weighted", scores=fit_verifier(P, Z, y, seed=s).score(te.P, te.Z)).mean()
            for s in [0, 1, 2]]
    rows[name + " majority"] = 100 * selection_hits(te, "majority").mean()
    rows[name + " LVPP"] = float(np.mean(accs))
    print(f"{name:14s} pool {len(y)} cands | majority {rows[name+' majority']:.2f} -> {np.mean(accs):.2f} +/- {np.std(accs):.2f}")
EXPECTED = {"distilled r32 majority": 62.17, "distilled r32 LVPP": 63.25,
            "few-shot CoT majority": 74.53, "few-shot CoT LVPP": 76.45,
            "Math-Instruct majority": 87.95, "Math-Instruct LVPP": 88.12}
report(rows, EXPECTED, list(EXPECTED), tol=0.3).to_csv(f"{RESULTS}/table4.csv", index=False)
print("\nDistilled r16 (K = 6: 63.46 -> 66.34) was a single run whose cache is not part of this release.")

In [ ]:
#@title Part B - Math-Instruct with the three-layer recipe (3,000-question pool)
if os.path.exists(PATHS["mi_test"]):
    P, Z, y = dict_B_as_pool(PATHS["mi_pool"])
    te = load_dict_B(PATHS["mi_test"])
    accs = [100 * selection_hits(te, "weighted", scores=fit_verifier(P, Z, y, seed=s).score(te.P, te.Z)).mean() for s in [0, 1, 2]]
    print(f"Math-Instruct, 3-layer: majority {100*selection_hits(te,'majority').mean():.2f} | "
          f"log-prob tie-break {100*selection_hits(te,'majority_lp').mean():.2f} -> {np.mean(accs):.2f} "
          f"(original run: 86.73 / 86.81 -> 87.69)")

In [ ]:
#@title Part C - OPTIONAL: train the LoRA-distilled generator and capture it (GPU, many hours)
RUN_DISTILL = False
if RUN_DISTILL:
    import subprocess
    subprocess.run([sys.executable, "-m", "lvpp.distill", "--rank", "32", "--out", f"{CACHE}/adapters/distill_r32"], check=True)
    from lvpp.capture import FewShotCapture, FewShotConfig
    from lvpp.data import gsm8k
    cap = FewShotCapture(FewShotConfig(adapter=f"{CACHE}/adapters/distill_r32", zero_shot_prompt=True))
    cap.build_pool(gsm8k("train"), f"{CACHE}/distilled_r32_ml/pool.pt", n_pool=800)
    cap.build_test(gsm8k("test"), f"{CACHE}/distilled_r32_ml/testfeat.pt")